# Crescimento, Interpolação e Extrapolação Populacional — Método Aritmético (Linear)
### Todos os Municípios de MG (1970-2025)

## Célula 1 — Imports

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

from utilitarios_populacional import (
    COLUNA_MUNICIPIO,
    COLUNA_CODIGO_IBGE,
    POPULACAO_MINIMA_VALIDA,
    calcular_taxa_aritmetica,
    estimar_populacao_aritmetica,
)

## Célula 2 — Configuração Geral

In [2]:
CAMINHO_ARQUIVO_ENTRADA = Path('/home/anna/tcc/Seleção e Transformação/População - Faixa Etária/tabela_populacao_faixa_etaria.csv')
CAMINHO_ARQUIVO_SAIDA = Path('/home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/populacao_interpolacao_extrapolacao_linear.csv')

LISTA_ANOS_CENSITARIOS = [1970, 1980, 1991, 2000, 2010, 2022]
ANO_INICIO = 1970
ANO_FIM = 2025
LISTA_ANOS_COMPLETOS = list(range(ANO_INICIO, ANO_FIM + 1))

## Leitura e Validação dos Anos Censitários

Confere se todas as colunas `TOTAL_<ano censitário>` esperadas realmente existem no arquivo de entrada, e se correspondem exatamente aos anos censitários definidos em `LISTA_ANOS_CENSITARIOS`.

In [3]:
tabela_populacao = pd.read_csv(CAMINHO_ARQUIVO_ENTRADA)

lista_colunas_totais = [f"TOTAL_{ano}" for ano in LISTA_ANOS_CENSITARIOS]

colunas_faltantes = [coluna for coluna in lista_colunas_totais if coluna not in tabela_populacao.columns]
if colunas_faltantes:
    raise ValueError(
        f"As seguintes colunas de anos censitários não foram encontradas no arquivo de entrada: {colunas_faltantes}. "
        f"Colunas TOTAL_* disponíveis: {[c for c in tabela_populacao.columns if c.startswith('TOTAL_')]}"
    )

anos_encontrados = []

for coluna in tabela_populacao.columns:
    if coluna.startswith("TOTAL_"):
        ano_texto = coluna.replace("TOTAL_", "")
        ano_inteiro = int(ano_texto)
        anos_encontrados.append(ano_inteiro)

anos_disponiveis_no_arquivo = sorted(anos_encontrados)
print("Anos censitários esperados  :", LISTA_ANOS_CENSITARIOS)
print("Anos censitários no arquivo :", anos_disponiveis_no_arquivo)

assert LISTA_ANOS_CENSITARIOS == anos_disponiveis_no_arquivo, (
    "Os anos censitários definidos em LISTA_ANOS_CENSITARIOS não coincidem "
    "exatamente com os anos TOTAL_* disponíveis no arquivo de entrada. "
    "Revise a configuração antes de prosseguir."
)

print("\nCorreto — todos os anos censitários conferem com o arquivo de entrada.")

# Índice = Código IBGE (mantido também como coluna via drop=False). Assim,
# qualquer join/concat feito por índice mais adiante (interpolação,
# extrapolação, exportação) está alinhado pela chave de município de
# verdade, e não pela posição da linha — resiliente a qualquer filtro ou
# reordenação que venha a ser introduzido no meio do pipeline.
tabela_totais = (
    tabela_populacao[[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE] + lista_colunas_totais]
    .copy()
    .set_index(COLUNA_CODIGO_IBGE, drop=False)
)
tabela_totais.head()

Anos censitários esperados  : [1970, 1980, 1991, 2000, 2010, 2022]
Anos censitários no arquivo : [1970, 1980, 1991, 2000, 2010, 2022]

Correto — todos os anos censitários conferem com o arquivo de entrada.


,MUNICÍPIOS,CÓDIGO IBGE,TOTAL_1970,TOTAL_1980,TOTAL_1991,TOTAL_2000,TOTAL_2010,TOTAL_2022
CÓDIGO IBGE,,,,,,,,
3100104,abadia dos dourados,3100104,4848,3936,3252,3105,3676,3111
3100203,abaete,3100203,10206,9926,10560,10947,11171,11474
3100302,abre campo,3100302,9779,9375,9131,6610,7238,6903
3100401,acaiaca,3100401,1699,1822,1937,1875,1881,2012
3100807,aguanil,3100807,1312,1560,1576,1753,1938,2196


## Checagem de Valores Não Positivos

Diferente do método geométrico (que precisa de população > 0 para dividir e extrair raiz), o método aritmético não tem essa restrição matemática — a taxa `k_a` é uma diferença dividida pelo intervalo de anos, nunca pela população. Esta checagem é mantida apenas como diagnóstico de qualidade dos dados de entrada (população <= 0 é sempre suspeito, independente do método).

In [4]:
municipios_com_valor_nao_positivo = tabela_totais[
    (tabela_totais[lista_colunas_totais] <= 0).any(axis=1)
]

if len(municipios_com_valor_nao_positivo) > 0:
    print(f"ATENÇÃO: {len(municipios_com_valor_nao_positivo)} município(s) com população <= 0 em algum ano censitário:")
    display(municipios_com_valor_nao_positivo[[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE] + lista_colunas_totais])
else:
    print("Nenhum município com população <= 0 em nenhum ano censitário.")

ATENÇÃO: 131 município(s) com população <= 0 em algum ano censitário:


,MUNICÍPIOS,CÓDIGO IBGE,TOTAL_1970,TOTAL_1980,TOTAL_1991,TOTAL_2000,TOTAL_2010,TOTAL_2022
CÓDIGO IBGE,,,,,,,,
3101631,alfredo vasconcelos,3101631,0,0,467,2581,2443,3471
3102050,alto caparao,3102050,0,0,385,2368,2135,2880
3102852,angelandia,3102852,0,0,505,3970,3475,3916
3103751,arapora,3103751,0,0,521,2715,2470,4246
3104452,aricanduva,3104452,0,0,294,2149,2020,2317
...,...,...,...,...,...,...,...,...
3170651,vargem grande do rio pardo,3170651,0,0,365,2362,1959,2248
3170750,varjao de minas,3170750,0,0,443,2420,2398,3354
3171030,verdelandia,3171030,0,0,517,3865,3546,3762


## Taxas de Crescimento (1970-2022) — Comparação Aritmética / Geométrica / Exponencial

Aritmética: crescimento por um valor absoluto constante por ano (reta).

Geométrica: crescimento a uma taxa percentual constante, com juros compostos discretos (ano a ano).

Exponencial: crescimento a uma taxa instantânea constante, com juros compostos contínuos.

In [5]:
populacao_inicial = tabela_totais[f"TOTAL_{LISTA_ANOS_CENSITARIOS[0]}"]
populacao_final = tabela_totais[f"TOTAL_{LISTA_ANOS_CENSITARIOS[-1]}"]
numero_de_anos = LISTA_ANOS_CENSITARIOS[-1] - LISTA_ANOS_CENSITARIOS[0]

# Município com população inicial 0 (ex.: ainda não emancipado em 1970)
# tornaria TAXA_GEOMETRICA/TAXA_EXPONENCIAL infinitas por divisão/log de
# zero — aqui só a título de comparação informativa, não usada em
# nenhum cálculo posterior, mas ainda assim vale não gerar inf.
populacao_inicial_valida = populacao_inicial.where(populacao_inicial >= POPULACAO_MINIMA_VALIDA)

taxa_crescimento_aritmetico = (populacao_final - populacao_inicial) / (numero_de_anos * populacao_inicial_valida)
taxa_crescimento_geometrico = (populacao_final / populacao_inicial_valida) ** (1 / numero_de_anos) - 1
taxa_crescimento_exponencial = (1 / numero_de_anos) * np.log(populacao_final / populacao_inicial_valida)

tabela_taxas_crescimento = pd.DataFrame({
    COLUNA_MUNICIPIO: tabela_totais[COLUNA_MUNICIPIO],
    COLUNA_CODIGO_IBGE: tabela_totais[COLUNA_CODIGO_IBGE],
    "TAXA_ARITMETICA": taxa_crescimento_aritmetico,
    "TAXA_GEOMETRICA": taxa_crescimento_geometrico,
    "TAXA_EXPONENCIAL": taxa_crescimento_exponencial,
})

tabela_taxas_crescimento.head()

,MUNICÍPIOS,CÓDIGO IBGE,TAXA_ARITMETICA,TAXA_GEOMETRICA,TAXA_EXPONENCIAL
CÓDIGO IBGE,,,,,
3100104,abadia dos dourados,3100104,-0.006890,-0.008495,-0.008531
3100203,abaete,3100203,0.002389,0.002255,0.002252
3100302,abre campo,3100302,-0.005656,-0.006675,-0.006698
3100401,acaiaca,3100401,0.003543,0.003257,0.003252
3100807,aguanil,3100807,0.012957,0.009955,0.009905


## Interpolação Aritmética (Linear)

$$k_a = \dfrac{P^{final} - P^{inicial}}{t} \qquad P^t = P^{inicial} + k_a \cdot t$$

In [6]:
def interpolar_aritmetica(tabela, colunas_censitarias, anos_censitarios, anos_completos):

    P = tabela[colunas_censitarias].copy()
    P.columns = anos_censitarios

    resultado = pd.DataFrame(index=P.index, columns=anos_completos, dtype=float)

    for ano in anos_censitarios:
        resultado[ano] = P[ano]

    for i in range(len(anos_censitarios) - 1):
        ano_inicial = anos_censitarios[i]
        ano_final = anos_censitarios[i + 1]

        populacao_inicial = P[ano_inicial]   # P^inicial da Fórmula 1
        populacao_final = P[ano_final]       # P^final da Fórmula 1
        tempo_total = ano_final - ano_inicial  # "t" da Fórmula 1 (intervalo entre os censos)

        # Fórmula 1
        k_a = calcular_taxa_aritmetica(populacao_inicial, populacao_final, tempo_total)

        # Preenche cada ano dentro deste intervalo com a Fórmula 2
        anos_intermediarios = [ano for ano in anos_completos if ano_inicial < ano < ano_final]
        for ano in anos_intermediarios:
            tempo_decorrido = ano - ano_inicial  # "t" da Fórmula 2 (desde o início do intervalo)
            resultado[ano] = estimar_populacao_aritmetica(populacao_inicial, k_a, tempo_decorrido)

    return resultado

tabela_interpolacao_linear = interpolar_aritmetica(
    tabela_totais, lista_colunas_totais, LISTA_ANOS_CENSITARIOS, LISTA_ANOS_COMPLETOS
)
tabela_interpolacao_linear.columns = [f"TOTAL_{ano}" for ano in LISTA_ANOS_COMPLETOS]
tabela_interpolacao_linear.head()

,TOTAL_1970,TOTAL_1971,TOTAL_1972,TOTAL_1973,TOTAL_1974,TOTAL_1975,TOTAL_1976,TOTAL_1977,TOTAL_1978,TOTAL_1979,...,TOTAL_2016,TOTAL_2017,TOTAL_2018,TOTAL_2019,TOTAL_2020,TOTAL_2021,TOTAL_2022,TOTAL_2023,TOTAL_2024,TOTAL_2025
CÓDIGO IBGE,,,,,,,,,,,,,,,,,,,,,
3100104,4848,4756.8,4665.6,4574.4,4483.2,4392.0,4300.8,4209.6,4118.4,4027.2,...,3393.5,3346.416667,3299.333333,3252.25,3205.166667,3158.083333,3111,NaN,NaN,NaN
3100203,10206,10178.0,10150.0,10122.0,10094.0,10066.0,10038.0,10010.0,9982.0,9954.0,...,11322.5,11347.750000,11373.000000,11398.25,11423.500000,11448.750000,11474,NaN,NaN,NaN
3100302,9779,9738.6,9698.2,9657.8,9617.4,9577.0,9536.6,9496.2,9455.8,9415.4,...,7070.5,7042.583333,7014.666667,6986.75,6958.833333,6930.916667,6903,NaN,NaN,NaN
3100401,1699,1711.3,1723.6,1735.9,1748.2,1760.5,1772.8,1785.1,1797.4,1809.7,...,1946.5,1957.416667,1968.333333,1979.25,1990.166667,2001.083333,2012,NaN,NaN,NaN
3100807,1312,1336.8,1361.6,1386.4,1411.2,1436.0,1460.8,1485.6,1510.4,1535.2,...,2067.0,2088.500000,2110.000000,2131.50,2153.000000,2174.500000,2196,NaN,NaN,NaN


## Extrapolação Aritmética (Linear)

$$k_a^{recente} = \dfrac{P_{2022}-P_{2010}}{12} \qquad P_t = P_{2022} + k_a^{recente}\cdot (t - 2022), \quad t > 2022$$

In [7]:
def extrapolar_aritmetica(tabela_interpolada, populacao_inicial_col, populacao_final_col,
                           ano_inicial, ano_final, anos_a_extrapolar):

    P_inicial = tabela_interpolada[populacao_inicial_col]
    P_final = tabela_interpolada[populacao_final_col]
    t = ano_final - ano_inicial

    k_a = calcular_taxa_aritmetica(P_inicial, P_final, t)

    extrapolacao = pd.DataFrame(index=tabela_interpolada.index)
    for ano in anos_a_extrapolar:
        extrapolacao[f"TOTAL_{ano}"] = P_final + k_a * (ano - ano_final)

    return extrapolacao, k_a

ULTIMOS_DOIS_CENSOS = LISTA_ANOS_CENSITARIOS[-2:]  # [2010, 2022]
ANOS_A_EXTRAPOLAR = [ano for ano in LISTA_ANOS_COMPLETOS if ano > LISTA_ANOS_CENSITARIOS[-1]]

tabela_extrapolacao_linear, taxa_aritmetica_extrapolacao = extrapolar_aritmetica(
    tabela_interpolacao_linear,
    populacao_inicial_col=f"TOTAL_{ULTIMOS_DOIS_CENSOS[0]}",
    populacao_final_col=f"TOTAL_{ULTIMOS_DOIS_CENSOS[1]}",
    ano_inicial=ULTIMOS_DOIS_CENSOS[0],
    ano_final=ULTIMOS_DOIS_CENSOS[1],
    anos_a_extrapolar=ANOS_A_EXTRAPOLAR,
)

print("Últimos dois censos usados na taxa de extrapolação:", ULTIMOS_DOIS_CENSOS)
print("Anos extrapolados                                 :", ANOS_A_EXTRAPOLAR)
tabela_extrapolacao_linear.head()

Últimos dois censos usados na taxa de extrapolação: [2010, 2022]
Anos extrapolados                                 : [2023, 2024, 2025]


,TOTAL_2023,TOTAL_2024,TOTAL_2025
CÓDIGO IBGE,,,
3100104,3063.916667,3016.833333,2969.75
3100203,11499.250000,11524.500000,11549.75
3100302,6875.083333,6847.166667,6819.25
3100401,2022.916667,2033.833333,2044.75
3100807,2217.500000,2239.000000,2260.50


## Interpolação + Extrapolação Aritmética (1970-2025)

Combina os resultados das duas etapas anteriores em uma única série contínua por município, do primeiro censo (1970) até o último ano definido em `ANO_FIM`.

In [8]:
tabela_populacao_completa = tabela_interpolacao_linear.copy()

for coluna in tabela_extrapolacao_linear.columns:
    tabela_populacao_completa[coluna] = tabela_populacao_completa[coluna].fillna(
        tabela_extrapolacao_linear[coluna]
    )

colunas_ordenadas = [f"TOTAL_{ano}" for ano in LISTA_ANOS_COMPLETOS]
tabela_populacao_completa = tabela_populacao_completa[colunas_ordenadas]

tabela_populacao_completa.head()

,TOTAL_1970,TOTAL_1971,TOTAL_1972,TOTAL_1973,TOTAL_1974,TOTAL_1975,TOTAL_1976,TOTAL_1977,TOTAL_1978,TOTAL_1979,...,TOTAL_2016,TOTAL_2017,TOTAL_2018,TOTAL_2019,TOTAL_2020,TOTAL_2021,TOTAL_2022,TOTAL_2023,TOTAL_2024,TOTAL_2025
CÓDIGO IBGE,,,,,,,,,,,,,,,,,,,,,
3100104,4848,4756.8,4665.6,4574.4,4483.2,4392.0,4300.8,4209.6,4118.4,4027.2,...,3393.5,3346.416667,3299.333333,3252.25,3205.166667,3158.083333,3111,3063.916667,3016.833333,2969.75
3100203,10206,10178.0,10150.0,10122.0,10094.0,10066.0,10038.0,10010.0,9982.0,9954.0,...,11322.5,11347.750000,11373.000000,11398.25,11423.500000,11448.750000,11474,11499.250000,11524.500000,11549.75
3100302,9779,9738.6,9698.2,9657.8,9617.4,9577.0,9536.6,9496.2,9455.8,9415.4,...,7070.5,7042.583333,7014.666667,6986.75,6958.833333,6930.916667,6903,6875.083333,6847.166667,6819.25
3100401,1699,1711.3,1723.6,1735.9,1748.2,1760.5,1772.8,1785.1,1797.4,1809.7,...,1946.5,1957.416667,1968.333333,1979.25,1990.166667,2001.083333,2012,2022.916667,2033.833333,2044.75
3100807,1312,1336.8,1361.6,1386.4,1411.2,1436.0,1460.8,1485.6,1510.4,1535.2,...,2067.0,2088.500000,2110.000000,2131.50,2153.000000,2174.500000,2196,2217.500000,2239.000000,2260.50


## Arredondamento

População é uma contagem de pessoas, então não faz sentido manter casas decimais nos valores interpolados/extrapolados. Usa-se `np.ceil` (arredondamento para cima) em vez de arredondamento comum, para nunca subestimar a população de nenhum município em nenhum ano.

In [9]:
colunas_ano = [col for col in tabela_populacao_completa.columns if col.startswith("TOTAL_")]

tabela_populacao_completa[colunas_ano] = (
    np.ceil(tabela_populacao_completa[colunas_ano])
    .replace([np.inf, -np.inf], np.nan)
    .astype("Int64")
)
tabela_populacao_completa.head()

,TOTAL_1970,TOTAL_1971,TOTAL_1972,TOTAL_1973,TOTAL_1974,TOTAL_1975,TOTAL_1976,TOTAL_1977,TOTAL_1978,TOTAL_1979,...,TOTAL_2016,TOTAL_2017,TOTAL_2018,TOTAL_2019,TOTAL_2020,TOTAL_2021,TOTAL_2022,TOTAL_2023,TOTAL_2024,TOTAL_2025
CÓDIGO IBGE,,,,,,,,,,,,,,,,,,,,,
3100104,4848,4757,4666,4575,4484,4392,4301,4210,4119,4028,...,3394,3347,3300,3253,3206,3159,3111,3064,3017,2970
3100203,10206,10178,10150,10122,10094,10066,10038,10010,9982,9954,...,11323,11348,11373,11399,11424,11449,11474,11500,11525,11550
3100302,9779,9739,9699,9658,9618,9577,9537,9497,9456,9416,...,7071,7043,7015,6987,6959,6931,6903,6876,6848,6820
3100401,1699,1712,1724,1736,1749,1761,1773,1786,1798,1810,...,1947,1958,1969,1980,1991,2002,2012,2023,2034,2045
3100807,1312,1337,1362,1387,1412,1436,1461,1486,1511,1536,...,2067,2089,2110,2132,2153,2175,2196,2218,2239,2261


## Validação Final — Valores Ausentes e Valores Negativos

Diferente do método geométrico, o método aritmético não gera `NaN` por população zero — mas, por ser aditivo (não multiplicativo), pode projetar população **negativa** para municípios em queda populacional acentuada, principalmente na extrapolação (anos mais distantes do último censo). Esta célula confere as duas coisas.

In [10]:
colunas_ano = [f"TOTAL_{ano}" for ano in LISTA_ANOS_COMPLETOS]

total_ausentes = tabela_populacao_completa[colunas_ano].isna().sum().sum()
print(f"Total de valores ausentes na série {ANO_INICIO}-{ANO_FIM}: {total_ausentes}")

if total_ausentes > 0:
    linhas_com_ausente = tabela_populacao_completa[colunas_ano].isna().any(axis=1)
    municipios_com_ausente = tabela_totais.loc[linhas_com_ausente, COLUNA_MUNICIPIO].tolist()
    print("Municípios com valores ausentes:", municipios_com_ausente)
else:
    print("Série completa — nenhum valor ausente entre", ANO_INICIO, "e", ANO_FIM, ".")

linhas_com_negativo = (tabela_populacao_completa[colunas_ano] < 0).any(axis=1)
total_municipios_com_negativo = linhas_com_negativo.sum()
print(f"\nTotal de municípios com população negativa em algum ano da série: {total_municipios_com_negativo}")

if total_municipios_com_negativo > 0:
    municipios_com_negativo = tabela_totais.loc[linhas_com_negativo, COLUNA_MUNICIPIO].tolist()
    print("Municípios com população negativa em algum ano:", municipios_com_negativo)
    print("ATENÇÃO: valores negativos não fazem sentido demograficamente — considere truncar em 0 "
          "antes de usar esta série como denominador de taxas.")

Total de valores ausentes na série 1970-2025: 0
Série completa — nenhum valor ausente entre 1970 e 2025 .

Total de municípios com população negativa em algum ano da série: 0


## Exportação

In [11]:
tabela_saida = tabela_totais[[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE]].merge(
    tabela_populacao_completa, left_index=True, right_index=True
)

CAMINHO_ARQUIVO_SAIDA.parent.mkdir(parents=True, exist_ok=True)
tabela_saida.to_csv(CAMINHO_ARQUIVO_SAIDA, index=False)

print("Arquivo salvo em:", CAMINHO_ARQUIVO_SAIDA)
print("Shape:", tabela_saida.shape)
print(f"Período coberto: {ANO_INICIO}-{ANO_FIM} ({len(LISTA_ANOS_COMPLETOS)} anos)")
tabela_saida.head()

Arquivo salvo em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/populacao_interpolacao_extrapolacao_linear.csv
Shape: (853, 58)
Período coberto: 1970-2025 (56 anos)


,MUNICÍPIOS,CÓDIGO IBGE,TOTAL_1970,TOTAL_1971,TOTAL_1972,TOTAL_1973,TOTAL_1974,TOTAL_1975,TOTAL_1976,TOTAL_1977,...,TOTAL_2016,TOTAL_2017,TOTAL_2018,TOTAL_2019,TOTAL_2020,TOTAL_2021,TOTAL_2022,TOTAL_2023,TOTAL_2024,TOTAL_2025
CÓDIGO IBGE,,,,,,,,,,,,,,,,,,,,,
3100104,abadia dos dourados,3100104,4848,4757,4666,4575,4484,4392,4301,4210,...,3394,3347,3300,3253,3206,3159,3111,3064,3017,2970
3100203,abaete,3100203,10206,10178,10150,10122,10094,10066,10038,10010,...,11323,11348,11373,11399,11424,11449,11474,11500,11525,11550
3100302,abre campo,3100302,9779,9739,9699,9658,9618,9577,9537,9497,...,7071,7043,7015,6987,6959,6931,6903,6876,6848,6820
3100401,acaiaca,3100401,1699,1712,1724,1736,1749,1761,1773,1786,...,1947,1958,1969,1980,1991,2002,2012,2023,2034,2045
3100807,aguanil,3100807,1312,1337,1362,1387,1412,1436,1461,1486,...,2067,2089,2110,2132,2153,2175,2196,2218,2239,2261


## Comparação — Valor Original vs. Valor Interpolado (Anos Censitários)

Compara, apenas nos anos de censo (1970, 1980, 1991, 2000, 2010, 2022), o valor original do arquivo de entrada com o valor da série interpolada/arredondada. Nesses anos os dois deveriam coincidir (ou diferir no máximo por causa do `np.ceil`), servindo como checagem de sanidade da interpolação.

In [12]:
lista_comparacoes = []

for ano in LISTA_ANOS_CENSITARIOS:
    coluna_ano = f"TOTAL_{ano}"
    comparacao_ano = pd.DataFrame({
        COLUNA_MUNICIPIO: tabela_totais[COLUNA_MUNICIPIO],
        "ANO": ano,
        "POPULACAO_ORIGINAL": tabela_totais[coluna_ano].values,
        "POPULACAO_INTERPOLADA": tabela_saida[coluna_ano].values,
    })
    lista_comparacoes.append(comparacao_ano)

tabela_comparacao_censos = pd.concat(lista_comparacoes, ignore_index=True)

tabela_comparacao_censos["DIFERENCA"] = (
    tabela_comparacao_censos["POPULACAO_INTERPOLADA"] - tabela_comparacao_censos["POPULACAO_ORIGINAL"]
)

print(f"Total de divergências entre original e interpolado nos anos censitários: {(tabela_comparacao_censos['DIFERENCA'] != 0).sum()}")

tabela_comparacao_censos

Total de divergências entre original e interpolado nos anos censitários: 0


,MUNICÍPIOS,ANO,POPULACAO_ORIGINAL,POPULACAO_INTERPOLADA,DIFERENCA
0,abadia dos dourados,1970,4848,4848,0
1,abaete,1970,10206,10206,0
2,abre campo,1970,9779,9779,0
3,acaiaca,1970,1699,1699,0
4,aguanil,1970,1312,1312,0
...,...,...,...,...,...
5113,wenceslau braz,2022,1176,1176,0
5114,agua boa,2022,6214,6214,0
5115,agua comprida,2022,1038,1038,0
5116,aguas formosas,2022,9181,9181,0
